<a href="https://colab.research.google.com/github/AyuLestari0405/flask1/blob/main/regresi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# 1. Load dataset langsung dari link raw GitHub
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv"
df_raw = pd.read_csv(url)

# 2. Pilih kota tertentu, contoh: 'Kota Banda Aceh'
kota_pilihan = "Kota Banda Aceh"
df_filtered = df_raw[df_raw["Kota"] == kota_pilihan].dropna(
    subset=["KEMISKINAN", "bm_pendidikan", "bm_infrastruktur"]
).copy()

# Reset index agar rapi
df_filtered = df_filtered.reset_index(drop=True)
df_filtered["ID_Observasi"] = np.arange(1, len(df_filtered) + 1)

# 3. Variabel Independen (X) dan Dependen (y)
X = df_filtered[["bm_pendidikan", "bm_infrastruktur"]]
y = df_filtered["KEMISKINAN"]

# 4. Menambahkan Intercept / Konstanta
X_model = sm.add_constant(X)

# 5. Estimasi Model OLS
model = sm.OLS(y, X_model)
results = model.fit()

# 6. Menampilkan Ringkasan Hasil Regresi
print("\n")
print("=" * 70)
print(f" HASIL REGRESI LINEAR BERGANDA - {kota_pilihan}")
print("=" * 70)
print(results.summary())

# 7. Mengambil Koefisien Regresi dan Nilai R-Squared
intercept = results.params["const"]
coef_pendidikan = results.params["bm_pendidikan"]
coef_infrastruktur = results.params["bm_infrastruktur"]
r_squared = results.rsquared
adjusted_r_squared = results.rsquared_adj

# 8. Membuat Persamaan Regresi
persamaan = (
    f"Ŷ = {intercept:.3f}"
    f" + ({coef_pendidikan:.3f}) X1"
    f" + ({coef_infrastruktur:.3f}) X2"
)
print("\n")
print("=" * 70)
print(" PERSAMAAN REGRESI")
print("=" * 70)
print(persamaan)
print(f"\nR2 = {r_squared:.4f}")
print(f"Adjusted R2 = {adjusted_r_squared:.4f}")

# 9. Menghitung Nilai Prediksi
df_filtered["Prediksi"] = results.predict(X_model)
print("\n")
print("=" * 70)
print(" DATA AKTUAL & PREDIKSI")
print("=" * 70)
print(
    df_filtered[[
        "ID_Observasi",
        "Tahun",
        "bm_pendidikan",
        "bm_infrastruktur",
        "KEMISKINAN",
        "Prediksi",
    ]].round(2)
)

# 10. Membuat Grid untuk Bidang Regresi (Regression Plane)
x1 = np.linspace(
    df_filtered["bm_pendidikan"].min(),
    df_filtered["bm_pendidikan"].max(),
    40,
)
x2 = np.linspace(
    df_filtered["bm_infrastruktur"].min(),
    df_filtered["bm_infrastruktur"].max(),
    40,
)
X1, X2 = np.meshgrid(x1, x2)

# 11. Menghitung Nilai Y pada Bidang Regresi
Y = intercept + coef_pendidikan * X1 + coef_infrastruktur * X2

# 12. Membuat Figure 3D Menggunakan Plotly
fig = go.Figure()

# 13. Menampilkan Data Aktual (Scatter Plot 3D)
fig.add_trace(
    go.Scatter3d(
        x=df_filtered["bm_pendidikan"],
        y=df_filtered["bm_infrastruktur"],
        z=df_filtered["KEMISKINAN"],
        mode="markers",
        marker=dict(size=7, color="red", opacity=0.9),
        name="Data Aktual",
        text=[f"Tahun {t}" for t in df_filtered["Tahun"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "BM Pendidikan (X1): %{x}<br>"
            "BM Infrastruktur (X2): %{y}<br>"
            "Kemiskinan Aktual (Y): %{z}"
            "<extra></extra>"
        ),
    )
)

# 14. Menampilkan Bidang Regresi (Surface Plot 3D)
fig.add_trace(
    go.Surface(
        x=X1,
        y=X2,
        z=Y,
        opacity=0.60,
        colorscale="Viridis",
        name="Bidang Regresi",
        hovertemplate=(
            "BM Pendidikan: %{x:.2f}<br>"
            "BM Infrastruktur: %{y:.2f}<br>"
            "Prediksi Kemiskinan: %{z:.2f}"
            "<extra></extra>"
        ),
    )
)

# 15. Mengatur Tampilan Grafik
fig.update_layout(
    title={
        "text": (
            f"REGRESI LINEAR BERGANDA 3D - {kota_pilihan}"
            "<br>"
            f"<sup>{persamaan} | R2 = {r_squared:.3f}</sup>"
        ),
        "x": 0.5,
    },
    scene={
        "xaxis": {"title": "Belanja Pendidikan (X1)"},
        "yaxis": {"title": "Belanja Infrastruktur (X2)"},
        "zaxis": {"title": "Tingkat Kemiskinan (Y)"},
    },
    width=1000,
    height=750,
    margin={"l": 0, "r": 0, "b": 0, "t": 100},
    legend={"x": 0.02, "y": 0.98},
)

# 16. Menampilkan Grafik Interaktif
fig.show()



 HASIL REGRESI LINEAR BERGANDA - Kota Banda Aceh
                            OLS Regression Results                            
Dep. Variable:             KEMISKINAN   R-squared:                       0.324
Model:                            OLS   Adj. R-squared:                  0.155
Method:                 Least Squares   F-statistic:                     1.916
Date:                Mon, 12 Oct 2026   Prob (F-statistic):              0.209
Time:                        23:12:53   Log-Likelihood:                -30.057
No. Observations:                  11   AIC:                             66.11
Df Residuals:                       8   BIC:                             67.31
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------

Pengaruh Belanja Modal Pendidikan ($X_1$) dan Belanja Modal Infrastruktur ($X_2$) terhadap Tingkat Kemiskinan ($Y$)(Studi Kasus: Kota Banda Aceh)1.
Variabel Dependen ($Y$): KEMISKINAN (Tingkat kemiskinan daerah)Variabel Independen 1 ($X_1$): bm_pendidikan (Belanja modal sektor pendidikan untuk peningkatan kualitas SDM)Variabel Independen 2 ($X_2$): bm_infrastruktur (Belanja modal sektor infrastruktur untuk memperlancar konektivitas dan aktivitas ekonomi lokal)